Improving the Random Forest:

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

df = pd.read_csv("ml_df_test_bab.csv")

# Train : Training the model on data until 2018
df_train = df[df["year"] <= 2018]

# Prediction : Predicting the results of World Cup 2022
prediction = df[df['year'] == 2022]

y_train = df_train["result_target"]
y_test = prediction["result_target"]

# 4. Deine handverlesenen Spalten (Feature Selection)
relevant_columns = [
    'elo_diff', 'win_rate_diff', 'goal_diff_diff', 'form_diff', 
    'goals_per_match_diff', 'conceded_per_match_diff',
    'home_elo_before', 'away_elo_before',
    'home_total_win_rate_before', 'away_total_win_rate_before',
    'home_goal_diff_per_match_before', 'away_goal_diff_per_match_before',
    'home_last5_win_rate', 'away_last5_win_rate',
    'home_last5_goal_diff', 'away_last5_goal_diff',
    'home_tournaments_played_before', 'away_tournaments_played_before',
    'home_has_won_world_cup_before', 'away_has_won_world_cup_before',
    'home_is_defending_champion', 'away_is_defending_champion'
]

#Chosing the relevant columns for training ( excluding strings/ spoilers)
x_train = df_train[relevant_columns].select_dtypes(exclude=['object'])
x_test = prediction[relevant_columns].select_dtypes(exclude=['object'])

print("✅ Schritt 1: Daten sind geladen und bereit für den Wald!")

✅ Schritt 1: Daten sind geladen und bereit für den Wald!


1. Our first Random-Forest-Model:

In [ ]:
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import classification_report

# Empty Forest:
forest_test = RandomForestClassifier(
    n_estimators=100,   #Number of trees in the forest, more trees -> better results, but it takes longer
    #Parameters for every single tree in the forest:
    max_depth=5,
    class_weight='balanced',
    random_state=42
)

#Training the 100 trees in the forest 
print("Planting the 100 trees:")
forest_test.fit(x_train, y_train) #Fit gives every tree random columns to learn from 
                                  #-> forces trees to be different, to be "creative" and not lazy

#Using the trained forest to predict the 2022 cup
y_pred_forest = forest_test.predict(x_test)
print("Prediction completed!")

print("\n Classification Report for the Random Forest model: \n" , classification_report(y_test, y_pred_forest))


Planting the 100 trees:
Prediction completed!

 Classification Report for the Random Forest model: 
               precision    recall  f1-score   support

     AwayWin       0.48      0.65      0.55        20
        Draw       0.24      0.27      0.25        15
     HomeWin       0.70      0.48      0.57        29

    accuracy                           0.48        64
   macro avg       0.47      0.47      0.46        64
weighted avg       0.52      0.48      0.49        64



Comparing it with our best Decision-Tree so far:

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

#These are the best parameters we found through the GridSearchCV in the previous notebook
#for the best decision tree model:
best_tree = DecisionTreeClassifier(
    max_depth=3,                 
    min_samples_split=2,         
    min_samples_leaf=1,          
    class_weight='balanced',     
    random_state=42
)


best_tree.fit(x_train, y_train)
y_pred_best_tree = best_tree.predict(x_test)

print(classification_report(y_test, y_pred_best_tree))

#COMPARING TO THE RANDOM FOREST MODEL:
print("\n Classification Report for the Random Forest model: \n" , classification_report(y_test, y_pred_forest))

              precision    recall  f1-score   support

     AwayWin       0.54      0.65      0.59        20
        Draw       0.50      0.20      0.29        15
     HomeWin       0.65      0.76      0.70        29

    accuracy                           0.59        64
   macro avg       0.56      0.54      0.53        64
weighted avg       0.58      0.59      0.57        64


 Classification Report for the Random Forest model: 
               precision    recall  f1-score   support

     AwayWin       0.48      0.65      0.55        20
        Draw       0.24      0.27      0.25        15
     HomeWin       0.70      0.48      0.57        29

    accuracy                           0.48        64
   macro avg       0.47      0.47      0.46        64
weighted avg       0.52      0.48      0.49        64



Right now the Decision-Tree has a better prediction accuracy with 59% than our Random-Forest with 48%.

So we are trying to improve our Random-Forest now:
What we are gonna do is, letting an algorithm choose which parameters create the best results for the model (just like we did it with the decision tree before)



In [11]:
forest_paramters = {
    #How many trees in the forest EFFICIENTLY? (More trees -> better results, but it takes longer)
    'n_estimators': [100, 200, 300],

    #Sometimes it is better to force the model to be creative by limitting the number of features it can learn from at each split,
    #because it forces the tree to be creative and not lazy (by chosing the same relevant features all the time), but only sometimes, so we will test it:
    "max_features": ['sqrt', 'log2', None], 

    'max_depth': [3, 4, 5, 6, 7],
    'min_samples_split': [2, 5, 10],

    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

empty_forest = RandomForestClassifier(random_state=42)

forest_roboter = GridSearchCV(
    estimator=empty_forest,        # Our empty forest
    param_grid=forest_paramters,   # The paramters we want to test 
    scoring='f1_macro',            # Which score to optimize for?
    cv=5,                          # 5-fache Kreuzvalidierung (schützt vor Overfitting)
    n_jobs=-1                      #How many CPU cores to use? 
)

# 3. Den Roboter starten (Hier passiert die echte Arbeit!)
forest_roboter.fit(x_train, y_train)

# 4. Das Sieger-Rezept ausdrucken
print("The chosen parameters for the best random forest are: ")
print(forest_roboter.best_params_)

# 5. Den besten Wald aus dem Rucksack holen und die WM 2022 tippen lassen
best_forest = forest_roboter.best_estimator_
y_pred_best_forest = best_forest.predict(x_test)

# 6. Das finale Zeugnis ausdrucken
print("Classification Report for the best random forest model: \n" , classification_report(y_test, y_pred_best_forest))



=== DIE OPTIMALEN WALD-EINSTELLUNGEN ===
{'class_weight': 'balanced', 'max_depth': 7, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}

🌲 KLAUSUR-ERGEBNIS: DER OPTIMIERTE RANDOM FOREST
              precision    recall  f1-score   support

     AwayWin       0.44      0.55      0.49        20
        Draw       0.19      0.27      0.22        15
     HomeWin       0.72      0.45      0.55        29

    accuracy                           0.44        64
   macro avg       0.45      0.42      0.42        64
weighted avg       0.51      0.44      0.46        64



The accuracy is still really bad and worse than our best Decision-Tree model.
If we look closer to the chosen parameters, we can see that the model did the mistake of overfitting. The chosen "max_depth" is at 7 and is way too detailed for the prediction of the World-Cup 2022 (the max_depth parameter of the Decision-Tree is 3). 

So trusting the GridSearch isnt a good choice for the Random-Forest.

2. Try of Improvement:
- Orienting on the Best model of the Decision Tree and just chosing the other parameters with common sense and testing



In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

optimized_forest = RandomForestClassifier(
    n_estimators=300,            # 300 Trees to maximize accuracy
    max_depth=3,                 # depth of 3 was the best for a single tree, so we use it for all trees in the forest now
    max_features="sqrt",         # forcing the model to be creative by limitting the number of features it can learn from at each split,
    min_samples_leaf= 3,         # 
    class_weight="balanced",     # We saw at the decision tree that balancing the class-weight is the best for predicting the results
    random_state=42,
    n_jobs=-1                    
)

optimized_forest.fit(x_train, y_train)

y_pred_optimized_forest = optimized_forest.predict(x_test)

print("Classification Report for the optimized random forest model: \n" , classification_report(y_test, y_pred_optimized_forest))




Classification Report for the optimized random forest model: 
               precision    recall  f1-score   support

     AwayWin       0.50      0.70      0.58        20
        Draw       0.45      0.33      0.38        15
     HomeWin       0.76      0.66      0.70        29

    accuracy                           0.59        64
   macro avg       0.57      0.56      0.56        64
weighted avg       0.61      0.59      0.59        64



It was possible to improve the Random-Forest to an accuracy of 59% after experimenting with the parameters and using one of the best possible combinations.

So overall, the best accuracy we could archive was the same as the accuracy of the Decision Tree with 59%